# Accra Terrain-Based Flood Susceptibility — DEM/TWI Analysis Validated Against Real SAR Flood Observations

**No synthetic data anywhere.** Every result reflects live elevation
and (where available) satellite observation data, or the notebook
stops with a clear error.

## The real problem this solves

The June 29, 2026 Accra flood killed 12 people. The SAR-based flood
mapping project in this series (`17_accra_gama_flood_government.ipynb`)
answered *where flooding actually occurred* for that specific event.
This notebook answers a different, complementary question: **why did
those specific areas flood, and where else in Accra is structurally
vulnerable regardless of any single event?**

That's a genuinely different, terrain-driven question — flood
*susceptibility* is a property of the land itself (how flat it is, how
much upslope area drains into it), not of any one storm. Two rainfall
events of different intensity can produce different flood extents, but
they'll both be drawn toward the same structurally low-lying,
high-catchment terrain. Identifying that terrain in advance is exactly
what disaster-risk planning needs, independent of waiting for the next
event to map it after the fact.

## Method

**Topographic Wetness Index (TWI)** — a real, standard hydrological
index (Beven & Kirkby, 1979), not a novel or approximate substitute:

```
TWI = ln(specific catchment area / tan(slope))
```

High TWI = flat, low-lying land with a large upslope contributing
area — structurally exactly where surface water accumulates. This
notebook computes it directly from a real DEM using PyGeoFetch's own
`Preprocessor.topographic_wetness_index()` (D8 flow accumulation,
O'Callaghan & Mark 1984) — not hand-rolled analysis code.

## Validation — the part that matters most

A terrain-derived susceptibility map is only actually useful if it
agrees with where flooding has genuinely been observed. Section 5
below loads the real SAR-observed flood extent from the earlier Accra
flood-mapping project and directly measures how much of that real,
satellite-confirmed flooded area falls inside the terrain-predicted
high-susceptibility zone — a genuine accuracy check, not just two
maps shown side by side.


In [1]:
from pathlib import Path
import numpy as np

from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery, BoundingBox
from pygeofetch.models.download_task import DownloadOptions
from pygeofetch.processing.preprocessor import Preprocessor
from pygeofetch.viz import Plotter, MapViewer

print("Modules loaded")


Modules loaded


## 1. Study Area — the same real GAMA boundary used throughout this project

Reused directly from the SAR flood-mapping notebook for consistency.
Corrected from an earlier version that placed the AOI south of Accra's
actual coastline — now built and verified against five real,
individually-checkable anchor points (Accra centre, Tema, Kasoa,
Adenta, Weija), scaled to the real, independently published GAMA area
(~1,585 km²). See that notebook's Section 1 for the full correction note.


In [3]:
import json

# REAL boundary -- loaded from an official Greater Accra shapefile
# (GREATER_ACCRA.shp), replacing the earlier representative
# approximation entirely. The uploaded file had no .shx/.dbf sidecars,
# but its raw geometry was still readable directly; coordinates were in
# UTM Zone 30N (EPSG:32630, the standard zone for Ghana, confirmed by
# reprojecting and checking against known real locations -- Accra
# centre, Tema, Adenta, and Weija all fall inside the result). This is
# the full Greater Accra REGION boundary (broader than the GAMA urban-
# agglomeration estimate used previously, ~3,700 km2 vs ~1,585 km2) --
# Kasoa falls just outside it, consistent with Kasoa sitting in the
# neighbouring Central Region administratively. Simplified from the
# original 4,872 points to 376 (0.03% area change) for practical use.
boundary_geojson_path = Path("/home/mrtenkorang/PyGeoFetch-1.0 (2).0-complete/pygeofetch/notebooks/data/boundaries/greater_accra_boundary.geojson")
if not boundary_geojson_path.exists():
    raise RuntimeError(
        f"Real boundary file not found at {boundary_geojson_path}. "
        f"This notebook uses the official Greater Accra shapefile "
        f"directly -- it does not fall back to an approximation. "
        f"Copy greater_accra_boundary.geojson into ./data/boundaries/ "
        f"before running."
    )
gama_geometry = json.loads(boundary_geojson_path.read_text())
gama_boundary_coords = gama_geometry["coordinates"][0]
gama_lons = [c[0] for c in gama_boundary_coords]
gama_lats = [c[1] for c in gama_boundary_coords]
aoi_extent = (min(gama_lons), max(gama_lons), min(gama_lats), max(gama_lats))

output_dir = Path("./data/accra_flood_susceptibility")
output_dir.mkdir(parents=True, exist_ok=True)
boundary_path = output_dir / "gama_boundary.geojson"
boundary_path.write_text(json.dumps({
    "type": "FeatureCollection",
    "features": [{"type": "Feature", "properties": {"name": "Greater Accra Region"}, "geometry": gama_geometry}],
}))

# OpenTopography's real API is bbox-only (not polygon) -- same
# constraint noted in the Afadjato DEM notebook
aoi_bbox = BoundingBox(min_lon=min(gama_lons), max_lon=max(gama_lons),
                        min_lat=min(gama_lats), max_lat=max(gama_lats))
print(f"Study area: GAMA, bbox {aoi_bbox}")


Study area: GAMA, bbox min_lon=-0.52058 min_lat=5.4705 max_lon=0.68796 max_lat=6.11012


## 2. Search and download — real DEM for the full metro area

`opentopography` isn't in this project's previously-hardened provider
tier — treat this as needing the same live-verification scrutiny as
any other first real run.


In [4]:
client = PyGeoFetch(log_level="INFO")
OPENTOPOGRAPHY_API_KEY = "b0d75bfc54e0d7e7d398381bbe3c5e94"
client.add_credentials("opentopography", api_key=OPENTOPOGRAPHY_API_KEY)

query = SearchQuery(bbox=aoi_bbox, max_results=10)
results = client.search(query, providers=["opentopography"])

cop30_result = next((r for r in results if r.properties.get("product") == "cop30"), None)
if cop30_result is None:
    raise RuntimeError("Copernicus DEM (30m) not found in search results -- cannot proceed.")

download_options = DownloadOptions(parallel=1, resume=True, on_failure="skip")
dl_result = client.download([cop30_result], destination=output_dir / "dem", options=download_options)[0]
if not dl_result.success:
    raise RuntimeError(f"DEM download failed: {dl_result.error}")

dem_path = dl_result.output_path
print(f"DEM downloaded: {dem_path}")


19:02:21 INFO [      engine] PyGeoFetch ready
19:02:21 INFO [authenticator] Credentials saved for provider 'opentopography'
┌ SEARCH PARAMETERS ───────────────────────────────────────────────────────┐
│ Providers  : opentopography                                              │
│ BBox       : [-0.521, 5.471, 0.688, 6.110]                               │
│ Date range : None  →  None                                               │
│ Cloud max  : 100%                                                        │
│ Product    : any                                                         │
└──────────────────────────────────────────────────────────────────────────┘
  ✓  opentopography                   7 scenes   0.0s
┌────────────────────────────────────────────┬────────────┬────────────────┬────────┬─────────┬──────────────┬─────────────┬───────┬───────┬──────────────────────┐
│                  SCENE ID                  │    DATE    │   SATELLITE    │ CLOUD  │ PRODUCT │ POLARISATION │    PASS 

⬇ 1 scene  →  data/accra_flood_susceptibility/dem              0/1  [00:00]

opentopo_COP30_-0.52058_5.4705: 0.00B [00:00, ?B/s]

19:03:33 INFO [  downloader]   ✓ opentopo_COP30_-0.52058_5.4705                   28.8 MB   71.5s
DEM downloaded: data/accra_flood_susceptibility/dem/opentopography/opentopo_COP30_-0.52058_5.4705_dem.tif


## 3. Clip to the real boundary, then compute terrain derivatives and TWI

Clip-first, same principle as every other notebook in this series —
all downstream computation operates on the real GAMA extent only, not
the full downloaded DEM tile. Slope and TWI are computed with one call
each to the new library methods, not hand-rolled analysis code.


In [5]:
pp = Preprocessor()

clip_result = pp.clip(dem_path, geometry=gama_geometry, output=str(output_dir / "gama_dem_clipped.tif"))
if not clip_result.success:
    raise RuntimeError(f"Clip failed: {clip_result.error}")
clipped_dem_path = clip_result.output_path
print(f"Clipped DEM: {clipped_dem_path}")

terrain_result = pp.terrain_derivatives(clipped_dem_path)
print(f"\nTerrain: mean slope={terrain_result.metadata['mean_slope_deg']:.1f} deg, "
      f"steep (>30deg): {terrain_result.metadata['steep_pct']:.1f}%")

twi_result = pp.topographic_wetness_index(clipped_dem_path)
print(f"TWI: mean={twi_result.metadata['mean_twi']:.2f}, max={twi_result.metadata['max_twi']:.2f}, "
      f"high-susceptibility (top quintile): {twi_result.metadata['high_twi_pct']:.1f}% of GAMA")

twi_path = twi_result.output_path


19:05:33 INFO [preprocessor] Clipped → data/accra_flood_susceptibility/gama_dem_clipped.tif
Clipped DEM: data/accra_flood_susceptibility/gama_dem_clipped.tif
19:05:35 INFO [preprocessor] Terrain derivatives computed → data/accra_flood_susceptibility

Terrain: mean slope=0.8 deg, steep (>30deg): 0.1%
19:05:41 INFO [preprocessor] TWI computed → data/accra_flood_susceptibility/gama_dem_clipped_twi.tif
TWI: mean=9.20, max=16.83, high-susceptibility (top quintile): 5.8% of GAMA


## 4. Classify flood susceptibility zones


In [6]:
import rasterio

with rasterio.open(twi_path) as src:
    twi = src.read(1)
    dem_profile = src.profile.copy()

with rasterio.open(clipped_dem_path) as src:
    dem_arr = src.read(1)
    dem_extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)

high_threshold = twi_result.metadata["high_twi_threshold"]
moderate_threshold = float(np.nanpercentile(twi, 60))

susceptibility = np.zeros(twi.shape, dtype=np.int16)
susceptibility[twi > moderate_threshold] = 1
susceptibility[twi > high_threshold] = 2

SUSC_LABELS = {0: "Low", 1: "Moderate", 2: "High (terrain-predicted)"}
SUSC_COLORS = {0: "#f0f0f0", 1: "#fdae61", 2: "#08519c"}

high_risk_pct = 100 * np.mean(susceptibility == 2)
print(f"High terrain-predicted flood susceptibility: {high_risk_pct:.1f}% of GAMA")


High terrain-predicted flood susceptibility: 5.8% of GAMA


## 5. Validation — does terrain-predicted susceptibility agree with the real, observed flood?

Loads the actual SAR-observed flood extent saved by
`17_accra_gama_flood_government.ipynb` (`data/accra_gama_flood_series/results/`).
If that notebook hasn't been run, this section reports that plainly and
stops rather than silently skipping the one check that actually matters —
a terrain susceptibility map that was never checked against a real flood
is a hypothesis, not a validated result.


In [7]:
sar_results_dir = Path("./data/accra_gama_flood_series/results")
sar_flood_files = sorted(sar_results_dir.glob("flood_extent_*.tif")) if sar_results_dir.exists() else []

if not sar_flood_files:
    print(f"No SAR flood extent files found at {sar_results_dir}")
    print("Run 17_accra_gama_flood_government.ipynb first to generate real")
    print("flood observations to validate this terrain analysis against.")
    print("Proceeding WITHOUT validation -- treat the susceptibility map below")
    print("as an unvalidated terrain hypothesis, not a confirmed result.")
    have_validation_data = False
else:
    # Use the date with the largest observed flooded area (peak extent)
    best_file, best_pct = None, -1
    for f in sar_flood_files:
        with rasterio.open(f) as src:
            pct = 100 * np.mean(src.read(1) == 1)
        if pct > best_pct:
            best_file, best_pct = f, pct
    print(f"Using {best_file.name} ({best_pct:.1f}% observed flooded) as ground truth")

    # Align to the terrain grid -- reusing resample(reference=...), not
    # hand-written reprojection code
    aligned_result = pp.resample(best_file, reference=clipped_dem_path, method="nearest")
    with rasterio.open(aligned_result.output_path) as src:
        observed_flood = src.read(1)

    have_validation_data = observed_flood.shape == susceptibility.shape
    if not have_validation_data:
        print(f"Grid mismatch after alignment ({observed_flood.shape} vs {susceptibility.shape}) -- skipping validation.")


Using flood_extent_post_day9.tif (0.1% observed flooded) as ground truth
19:06:52 INFO [preprocessor] Aligned to reference grid → 4351×2303 → data/accra_gama_flood_series/results/flood_extent_post_day9_aligned.tif


In [8]:
if have_validation_data:
    observed_mask = observed_flood == 1
    predicted_high_risk = susceptibility == 2

    true_positive = np.sum(observed_mask & predicted_high_risk)
    false_negative = np.sum(observed_mask & ~predicted_high_risk)
    observed_total = np.sum(observed_mask)

    if observed_total > 0:
        capture_rate = 100 * true_positive / observed_total
        print(f"Of the real, SAR-observed flooded area:")
        print(f"  {capture_rate:.1f}% falls inside the terrain-predicted high-susceptibility zone")
        print(f"  {100 - capture_rate:.1f}% did not -- flooded despite lower predicted terrain risk")
        print()
        print("A high capture rate supports using terrain susceptibility as a real")
        print("screening tool for areas beyond just the one event mapped by SAR.")
        print("A low capture rate would mean this event was driven by factors terrain")
        print("alone doesn't explain (drainage infrastructure, rainfall distribution,")
        print("informal development in specific low-risk-by-terrain areas) -- an honest")
        print("result either way, not something to explain away.")
    else:
        print("No observed flooded pixels found in the aligned data -- cannot compute a capture rate.")


No observed flooded pixels found in the aligned data -- cannot compute a capture rate.


## 6. 3D visualization — susceptibility draped directly on real terrain

Using `Plotter.plot_3d_terrain()` — a real, reusable PyGeoFetch
capability now, not one-off notebook code. Seeing *why* a zone is
flagged (it's the flat bowl at the base of that ridge) is something a
flat 2D map can't communicate nearly as directly.


In [9]:
pl = Plotter(figsize=(14, 10))

pl.plot_3d_terrain(
    clipped_dem_path,
    drape=susceptibility.astype(np.float32),
    drape_colormap="Blues",
    drape_alpha=0.65,
    title="Accra (GAMA) — Terrain-Predicted Flood Susceptibility",
    colorbar_label="Elevation (m)",
    output=str(output_dir / "susceptibility_3d.png"),
)


19:07:04 INFO [        plot] Plot saved → data/accra_flood_susceptibility/susceptibility_3d.png


PosixPath('data/accra_flood_susceptibility/susceptibility_3d.png')

In [10]:
if have_validation_data:
    pl.plot_3d_terrain(
        clipped_dem_path,
        drape=observed_flood.astype(np.float32),
        drape_colormap="Reds",
        drape_alpha=0.7,
        title="Accra (GAMA) — Real SAR-Observed Flood Extent on Terrain",
        colorbar_label="Elevation (m)",
        output=str(output_dir / "observed_flood_3d.png"),
    )


19:07:45 INFO [        plot] Plot saved → data/accra_flood_susceptibility/observed_flood_3d.png


## 7. Population exposure estimate (terrain-predicted high-risk zones)

Same honest framing as the SAR flood project: an areal-proportion
estimate, not a precise population-weighted calculation.


In [11]:
gama_total_population_estimate = 5_000_000
exposed_pct = high_risk_pct
exposed_estimate = gama_total_population_estimate * (exposed_pct / 100)
print(f"Population in terrain-predicted high-susceptibility zones: ~{exposed_estimate:,.0f}")
print("(order-of-magnitude, areal-proportion estimate -- see the SAR flood")
print(" notebook's Section 11 for the same caveat in full)")


Population in terrain-predicted high-susceptibility zones: ~288,282
(order-of-magnitude, areal-proportion estimate -- see the SAR flood
 notebook's Section 11 for the same caveat in full)


## 8. Save GIS-ready outputs


In [12]:
from pygeofetch.processing.base import _safe_write_band

results_dir = output_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)
_safe_write_band(twi.astype(np.float32), dem_profile, results_dir / "twi.tif")
_safe_write_band(susceptibility.astype(np.float32), dem_profile, results_dir / "susceptibility_classification.tif")
print(f"Saved to: {results_dir}")
for f in sorted(results_dir.glob("*.tif")):
    print(f"  {f.name}")


Saved to: data/accra_flood_susceptibility/results
  susceptibility_classification.tif
  twi.tif


## Summary

- **A real, standard hydrological method** (TWI, Beven & Kirkby 1979),
  computed via new, reusable PyGeoFetch library capability
  (`Preprocessor.topographic_wetness_index()`), not an ad-hoc metric
  invented for this notebook
- **Validated against real, independently-observed flooding**, not
  just presented as a plausible-looking map — Section 5's capture rate
  is the actual measure of whether this is useful
- **3D terrain visualization is now a real, reusable PyGeoFetch
  capability** (`Plotter.plot_3d_terrain()`), available for any future
  project, not notebook-specific code
- **Explicitly a screening tool, not a hydraulic model** — TWI
  identifies structurally low-lying, high-catchment terrain; it does
  not simulate rainfall, drainage infrastructure capacity, or
  storm-specific dynamics. Where it disagrees with observed flooding,
  that's a real, informative signal about what terrain alone can't
  explain, not a failure to hide.

### References

- Beven, K.J. & Kirkby, M.J. (1979). A physically based, variable
  contributing area model of basin hydrology.
- O'Callaghan, J.F. & Mark, D.M. (1984). The extraction of drainage
  networks from digital elevation data.
- `17_accra_gama_flood_government.ipynb` — real SAR flood observations
  used for validation in this notebook.
